### Import necessary libraries

In [118]:
import os                       # Execute tasks related to your operating system.

import pandas as pd             # Data handling.
import numpy as np              # numerical calculations.
import pickle                   # Save and load data to and from pickle files.
import altair as alt            # Visualize data
import seaborn as sns           # Visualize data
import matplotlib.pyplot as plt # Visualize data

from scipy import stats  # Correlation
from scipy.stats import pearsonr
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import time


### Options for display

In [119]:
# Pandas.
pd.set_option("display.max_rows", None)         # How many rows to display when printing a DataFrame? None shows all rows.
pd.set_option("display.max_columns", None)      # How many columns to display when printing a DataFrame? None shows all columns.
pd.set_option("display.max_info_columns", 100)  # How many rows to display for .info()?
pd.set_option("display.max_info_rows", 1000000) # How many columns to display for .info()?
pd.set_option("display.precision", 2)           # How many decimals to show when printing a DataFrame?
sns.set(style="whitegrid")

### Load dataset

In [120]:
with open('../../../Data/data/dc-scoped-df.pkl', 'rb') as pickle_file:
    dc_scoped_df = pickle.load(pickle_file)


# Access the DataFrame
df  =   dc_scoped_df['df_scoped_5']


In [121]:
# Checking the first few rows of the dataset
df.head()


,id,store_nbr,item_nbr,unit_sales,onpromotion,day,month,family,class,perishable,city,state,type_x,cluster,date,dcoilwtico,type_y,locale,locale_name,description,transferred,temperature_2m_max,salary_payment,magnitude
0,16375666,44,122095.0,9.0,None,2,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-02,95.14,None,None,None,None,None,15.6,False,NaN
1,16441082,44,122095.0,14.0,None,3,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-03,93.66,None,None,None,None,None,16.0,False,NaN
2,16509198,44,122095.0,5.0,None,4,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-04,NaN,None,None,None,None,None,16.5,False,NaN
3,16576981,44,122095.0,6.0,None,5,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-05,NaN,None,None,None,None,None,15.0,False,NaN
4,16640795,44,122095.0,14.0,None,6,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-06,93.12,None,None,None,None,None,15.1,False,NaN


### Rename features for clarity

In [122]:
# Convert 'date' to datetime
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Rename 'type_x' to 'type_store'
df = df.rename(columns={'type_x': 'type_store'})

# Rename 'dcoilwtico' to 'oil price'
df = df.rename(columns={'dcoilwtico': 'oil_price'})

# Rename 'type_y' to 'type holiday'
df = df.rename(columns={'type_y': 'type_holiday'})

# View the result
df.head(10)

,id,store_nbr,item_nbr,unit_sales,onpromotion,day,month,family,class,perishable,city,state,type_store,cluster,date,oil_price,type_holiday,locale,locale_name,description,transferred,temperature_2m_max,salary_payment,magnitude
0,16375666,44,122095.0,9.0,None,2,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-02,95.14,None,None,None,None,None,15.6,False,NaN
1,16441082,44,122095.0,14.0,None,3,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-03,93.66,None,None,None,None,None,16.0,False,NaN
2,16509198,44,122095.0,5.0,None,4,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-04,NaN,None,None,None,None,None,16.5,False,NaN
3,16576981,44,122095.0,6.0,None,5,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-05,NaN,None,None,None,None,None,15.0,False,NaN
4,16640795,44,122095.0,14.0,None,6,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-06,93.12,None,None,None,None,None,15.1,False,NaN
5,16703588,44,122095.0,8.0,None,7,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-07,93.31,None,None,None,None,None,13.3,False,NaN
6,16766464,44,122095.0,6.0,None,8,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-08,91.90,None,None,None,None,None,13.5,False,NaN
7,16828954,44,122095.0,16.0,None,9,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-09,91.36,None,None,None,None,None,14.0,False,NaN
8,16891247,44,122095.0,7.0,None,10,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-10,92.39,None,None,None,None,None,16.3,False,NaN
9,16958728,44,122095.0,19.0,None,11,2014-01,DAIRY,2124,1,Quito,Pichincha,A,5,2014-01-11,NaN,None,None,None,None,None,14.2,False,NaN


## 1. Memory reduction & Descriptive statistics

In [123]:
# Check the data types
print("Data Types of Columns:")
print(df.dtypes)

# Checking number of observations/rows
df_obs = df.shape[0]
print(df_obs)

# Select numerical, date and categorical columns
df_num = df.select_dtypes(include=[np.number, bool])  # Numerical columns
df_str = df.select_dtypes(include=['object', 'category'])  # Categorical columns'
df_date = df.select_dtypes(include=['datetime64[ns]'])  # Datetime columns

Data Types of Columns:
id                             int32
store_nbr                       int8
item_nbr                     float32
unit_sales                   float32
onpromotion                   object
day                             int8
month                       category
family                      category
class                          int16
perishable                      int8
city                        category
state                       category
type_store                  category
cluster                         int8
date                  datetime64[ns]
oil_price                    float64
type_holiday                  object
locale                        object
locale_name                   object
description                   object
transferred                   object
temperature_2m_max           float64
salary_payment                  bool
magnitude                    float64
dtype: object
1706418


### Reduce Memory

In [124]:
# Reduce the memory usage by casting string type to category type data and numerical data to their smallest container size.

df_num_names = df_num.columns.tolist()
df_str_names = df_str.columns.tolist()
df_str_date = df_date.columns.tolist()
df_names = df.columns.tolist()


df_reduced = df.copy(deep=False)

# String to category using "as type()"
# Pandas:
df_reduced[df_str_names] = (
    df_reduced[df_str_names]
    .astype('category')
)

# Numerical reduce with zip() This is a FOR loop where old, new are = to i, j 

for old, new in zip(['integer', 'float'], ['unsigned', 'float']):

    for s_col in df_reduced.select_dtypes(include=old).columns:
        
        df_reduced[s_col] = pd.to_numeric(df_reduced[s_col], downcast=new)


# check how much it reduced, using memory_usage from Pandas 

par1 = df.memory_usage().sum()/(1024**2)
par2 = df_reduced.memory_usage().sum()/(1024**2)

print(f"Size of original data frame               : {round(par1, 1)} MB.")
print(f"Size of imputed and downcasted data frame : {round(par2, 2)} MB.")
print(f"Reduced original data frame by factor of  : {round(par1/par2)}")


Size of original data frame               : 169.3 MB.
Size of imputed and downcasted data frame : 81.38 MB.
Reduced original data frame by factor of  : 2


### Check data consistency

In [125]:
# Select numerical, date and categorical columns
df_num = df_reduced.select_dtypes(include=[np.number, bool])  # Numerical columns
df_str = df_reduced.select_dtypes(include=['object', 'category'])  # Categorical columns'
df_date = df_reduced.select_dtypes(include=['datetime64[ns]'])  # Datetime columns

# Check the shape of these subsets to confirm they're selected correctly
print(f"The original data has {df_reduced.shape[1]} columns.")
print(f"The number of numerical columns is {df_num.shape[1]}.")
print(f"The number of categorical columns is {df_str.shape[1]}.")
print(f"The number of datetime columns is {df_date.shape[1]}.")
print(f"Total number of columns in all types: {df_num.shape[1] + df_str.shape[1] + df_date.shape[1]}")

The original data has 24 columns.
The number of numerical columns is 12.
The number of categorical columns is 11.
The number of datetime columns is 1.
Total number of columns in all types: 24


### Descriptive statistics numerical features

In [126]:
# Checking numerical variables
df_missing_num = (
        pd.DataFrame({
        'data_type': df_num.dtypes,
        'missing': df_num.isnull().sum()
        })
    # Add complete%
    .assign(
        complete_pct = lambda x: round(100 * (df_obs - x['missing']) / df_obs, 2)
    )

    # Remove variables that are complete.
    .query("complete_pct < 100")

    # Sort table by data completeness in descending order.
    .sort_values(by = 'complete_pct')
)

# Summary statistics for numerical features
summary_stats_num = df_num.describe()

# Display the results
print("Missing values:")
print(df_missing_num)
print("\nSummary statistics for numerical columns:")
print(summary_stats_num)

Missing values:
          data_type  missing  complete_pct
magnitude   float32  1697632          0.51
oil_price   float32   541999         68.24

Summary statistics for numerical columns:
             id  store_nbr  item_nbr  unit_sales       day     class  \
count  1.71e+06   1.71e+06  1.71e+06    1.71e+06  1.71e+06  1.71e+06   
mean   6.80e+07   4.74e+01  1.05e+06    1.14e+01  1.56e+01  2.14e+03   
std    3.16e+07   2.29e+00  4.18e+05    1.31e+01  8.79e+00  2.52e+01   
min    1.64e+07   4.40e+01  1.22e+05   -5.40e+01  1.00e+00  2.10e+03   
25%    4.04e+07   4.50e+01  8.38e+05    4.00e+00  8.00e+00  2.12e+03   
50%    6.71e+07   4.70e+01  1.16e+06    8.00e+00  1.60e+01  2.13e+03   
75%    9.48e+07   4.90e+01  1.24e+06    1.40e+01  2.30e+01  2.17e+03   
max    1.25e+08   5.10e+01  2.01e+06    1.41e+03  3.10e+01  2.18e+03   

       perishable   cluster  oil_price  temperature_2m_max  magnitude  
count    1.71e+06  1.71e+06   1.16e+06            1.71e+06    8786.00  
mean     1.00e+00  

### Descriptive statistics categorical features

In [127]:
# Checking categorical variables
df_missing_str = (
        pd.DataFrame({
        'data_type': df_str.dtypes,
        'missing': df_str.isnull().sum()
        })
    # Add complete%
    .assign(
        complete_pct = lambda x: round(100 * (df_obs - x['missing']) / df_obs, 2)
    )

    # Remove variables that are complete.
    .query("complete_pct < 100")

    # Sort table by data completeness in descending order.
    .sort_values(by = 'complete_pct')
)

# Summary statistics for numerical features
summary_stats_str = df_str.describe()

# Display the results
print("Missing values:")
print(df_missing_str)
print("\nSummary statistics for categorical columns:")
print(summary_stats_str)

Missing values:
             data_type  missing  complete_pct
type_holiday  category  1536522          9.96
locale        category  1536522          9.96
locale_name   category  1536522          9.96
description   category  1536522          9.96
transferred   category  1536522          9.96
onpromotion   category    93087         94.54

Summary statistics for categorical columns:
       onpromotion    month   family     city      state type_store  \
count      1613331  1706418  1706418  1706418    1706418    1706418   
unique           2       44        1        3          3          1   
top          False  2016-05    DAIRY    Quito  Pichincha          A   
freq       1517531    50488  1706418  1305278    1305278    1706418   

       type_holiday    locale locale_name description transferred  
count        169896    169896      169896      169896      169896  
unique            6         2           2          70           2  
top           Event  National     Ecuador    Carnaval    

## 2. Preprocess Data

In [128]:
def preprocess_data(df_str, df_num):
    
    # Step 1: Impute missing categorical columns in df_str with "NO_HOLIDAY"
    categorical_cols = ['type_holiday', 'locale', 'locale_name', 'description', 'transferred']
    
    for col in categorical_cols:
        if col in df_str.columns:
            # Convert the column to 'category' dtype if it's not already
            if not pd.api.types.is_categorical_dtype(df_str[col]):
                df_str[col] = df_str[col].astype('category')
            
            # Check if 'NO_HOLIDAY' is already in the categories, and add it if not
            if 'NO_HOLIDAY' not in df_str[col].cat.categories:
                df_str[col] = df_str[col].cat.add_categories('NO_HOLIDAY')
            
            # Fill missing values with 'NO_HOLIDAY'
            df_str[col] = df_str[col].fillna('NO_HOLIDAY')
    

     # Step 2: Merge df_num with df_date on 'date' to ensure the 'date' column is available
    df_num = df_num.merge(df_date[['date']], left_index=True, right_index=True, how='left')
        # Ensure 'date' is a valid datetime type and set it as the index for interpolation
    if 'date' in df_num.columns:
        df_num['date'] = pd.to_datetime(df_num['date'], errors='coerce')
        
        # Drop rows where 'date' is invalid (NaT) before interpolation
        df_num = df_num.dropna(subset=['date'])
        
        # Set 'date' as the index
        df_num.set_index('date', inplace=True)


     # Step 3: Interpolate missing Oil prices
        # Ensure 'oil_price' column is numeric before interpolation
        df_num['oil_price'] = pd.to_numeric(df_num['oil_price'], errors='coerce')

        # Interpolate 'oil_price' based on time
        if 'oil_price' in df_num.columns:
            df_num['oil_price'] = df_num['oil_price'].interpolate(method='time')


     # Step 4: Impute magnitude missing values

        # Ensure 'magnitude' column is numeric before imputing
        df_num['magnitude'] = pd.to_numeric(df_num['magnitude'], errors='coerce')

        # Impute 'magnitude' with 0's
        df_num['magnitude'].fillna(0, inplace=True)



        # Reset the index 
        df_num.reset_index(inplace=True)

    # Return the processed dataframes
    return df_str, df_num


# Apply preprocessing to the DataFrames
df_processed_str, df_processed_num = preprocess_data(df_str, df_num)

# Compare the original and processed shapes
pd.DataFrame({
    'original_str':     df_str.shape,
    'processed_str':    df_processed_str.shape,
    'original_num':     df_num.shape,
    'processed_num':    df_processed_num.shape,
})

# Concatenate the cleaned DataFrames horizontally (axis=1)
df_processed = pd.concat([df_processed_str, df_processed_num], axis=1)

# Drop rows where 'unit_sales' == 0 (i.e., no sales, only returns)
df_processed = df_processed[df_processed['unit_sales'] > 0]

# Checking number of observations/rows
df_obs = df_processed.shape[0]
print(df_obs) 

# Confirm the original and updated data frames have the same shape
print("Shape of original data frame:", df_reduced.shape)
print("Shape of processed data frame:", df_processed.shape)




C:\Users\jasmi\AppData\Local\Temp\ipykernel_27896\951526992.py:9: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if not pd.api.types.is_categorical_dtype(df_str[col]):


1706352
Shape of original data frame: (1706418, 24)
Shape of processed data frame: (1706352, 24)


## 3. Summary statistics for Processed dataframe (numerical features)

In [129]:
print("Summary Statistics for Processed dataframe:")
df_processed.describe() # Summary of processed dataframe


Summary Statistics for Processed dataframe:


,date,id,store_nbr,item_nbr,unit_sales,day,class,perishable,cluster,oil_price,temperature_2m_max,magnitude
count,1706352,1.71e+06,1.71e+06,1.71e+06,1.71e+06,1.71e+06,1.71e+06,1.71e+06,1.71e+06,1.71e+06,1.71e+06,1.71e+06
mean,2015-12-09 02:22:20.341499904,6.80e+07,4.74e+01,1.05e+06,1.14e+01,1.56e+01,2.14e+03,1.00e+00,1.24e+01,5.73e+01,1.65e+01,3.37e-02
min,2014-01-02 00:00:00,1.64e+07,4.40e+01,1.22e+05,1.00e+00,1.00e+00,2.10e+03,1.00e+00,5.00e+00,2.62e+01,1.22e+01,0.00e+00
25%,2015-02-01 00:00:00,4.04e+07,4.50e+01,8.38e+05,4.00e+00,8.00e+00,2.12e+03,1.00e+00,1.10e+01,4.49e+01,1.55e+01,0.00e+00
50%,2016-01-08 00:00:00,6.71e+07,4.70e+01,1.16e+06,8.00e+00,1.60e+01,2.13e+03,1.00e+00,1.40e+01,4.91e+01,1.64e+01,0.00e+00
75%,2016-10-23 00:00:00,9.48e+07,4.90e+01,1.24e+06,1.40e+01,2.30e+01,2.17e+03,1.00e+00,1.40e+01,5.90e+01,1.74e+01,0.00e+00
max,2017-08-15 00:00:00,1.25e+08,5.10e+01,2.01e+06,1.41e+03,3.10e+01,2.18e+03,1.00e+00,1.70e+01,1.08e+02,2.14e+01,7.80e+00
std,NaN,3.16e+07,2.29e+00,4.18e+05,1.31e+01,8.79e+00,2.52e+01,0.00e+00,3.38e+00,2.14e+01,1.42e+00,4.70e-01


### Check missing values Processed dataframe

In [130]:
df_processed_missing = (
        pd.DataFrame({
        'data_type': df_processed.dtypes,
        'missing': df_processed.isnull().sum()
        })
    # Add complete%
    .assign(
        complete_pct = lambda x: round(100 * (df_obs - x['missing']) / df_obs, 2)
    )

    # Remove variables that are complete.
    .query("complete_pct < 100")

    # Sort table by data completeness in descending order.
    .sort_values(by = 'complete_pct')
)

# Display the results
print("Missing values:")
print(df_processed_missing)

####we will handle on promotion later on

Missing values:
            data_type  missing  complete_pct
onpromotion  category    93085         94.54


In [131]:
# Save exercise

# Create dictionary 'df_scoped_processed' with objects that will be used in the next parts.
df_scoped_processed = {
    'df_processed':      df_processed,
    }



# Save dc_scoped_df as 'df_processed.pkl'
with open('../../data/df_processed.pkl', 'wb') as pickle_file:
    pickle.dump(df_scoped_processed, pickle_file)


